<a href="https://colab.research.google.com/github/flg37/Beng-Dissertation/blob/main/Model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###
## Model 1 Training
###

"""
Training script for a multimodal Mamba and ConvNext based battery SOH estimator.
This script:
1. Loads multimodal session-level features and smoothed SOH labels from Google Drive.
2. Synchronizes feature blocks with target labels and constructs a PyTorch Dataset.
3. Defines a ConvNeXt{Mamba{MLP multimodal regression architecture.
4. Trains an ensemble of models across multiple master seeds and run-specific seeds
using an AdamW optimizer and OneCycleLR learning rate schedule.
5. Applies early stopping based on validation MAPE and saves the best model weights
for each (seed, run) configuration.
6. Records ensemble performance per seed and writes a JSON summary to disk.
"""
import numpy as np
import torch
import pandas as pd
61
import pickle
import sys
import random
import os
from torch.utils.data import Dataset, DataLoader
import math
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
import gc
import json
import datetime
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn as nn
from mamba_ssm import Mamba
# ============================================================================
# 0. SET SEED FOR REPRODUCIBILITY
# ============================================================================
def set_seed(seed: int = 20) -> None:
"""
Fix random seeds to improve reproducibility of experiments.
This function sets seeds for Python’s built-in random module, NumPy, and
PyTorch. In this configuration, full CUDA determinism is not enforced in
order to retain hardware optimizations, but a fixed seed still controls
the bulk of stochastic operations (initialization, shuffling, etc.).
Args:
seed (int): Random seed used for all stochastic components.
"""
random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
# If full determinism is required, the following lines can be enabled.
# Note: enforcing deterministic algorithms can significantly reduce speed.
# torch.cuda.manual_seed_all(seed)
# torch.backends.cudnn.deterministic = True
# torch.use_deterministic_algorithms(True, warn_only=True)
# cuDNN benchmark is disabled to avoid data-dependent algorithm selection.
torch.backends.cudnn.benchmark = False
# ============================================================================
# 1. GOOGLE DRIVE CONFIGURATION & DATA LOADING
# ============================================================================
# Base folders for Google Drive in Colab
DRIVE_PREFIX = "/content/drive/MyDrive/Beng_Dissertation"
DRIVE_PREFIX2 = "/content/drive/MyDrive"
# Select GPU if available, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
62
# Paths to multimodal features and SOH label CSV in Google Drive
INPUT_DATA_PATH = f"{DRIVE_PREFIX}/multimodal_features_synced2.pkl"
LABELS_CSV_PATH = f"{DRIVE_PREFIX}/session_level_soh_smoothed_final2.csv"
print(f"Loading extracted features from {INPUT_DATA_PATH}...")
try:
with open(INPUT_DATA_PATH, "rb") as f:
# raw_features is a list of dictionaries, each containing multimodal inputs
raw_features = pickle.load(f)
print(f" Loaded {len(raw_features)} feature blocks.")
except Exception as e:
sys.exit(f"\n CRITICAL ERROR loading features: {e}")
print(f"Loading SOH labels from {LABELS_CSV_PATH}...")
try:
with open(LABELS_CSV_PATH, "rb") as f:
labels_df = pd.read_csv(f)
print(f" Loaded {len(labels_df)} target SOH labels.")
except Exception as e:
sys.exit(f"\n CRITICAL ERROR loading labels: {e}")
# ============================================================================
# 2. SYNCHRONIZE FEATURES WITH LABELS
# ============================================================================
print(" Merging features with target labels...")
# Handle both ’vehicle_id’ and ’vehicle_id_split’ column names
vid_col = "vehicle_id_split" if "vehicle_id_split" in labels_df.columns else "vehicle_id"
labels_df[vid_col] = labels_df[vid_col].astype(str)
labels_df["session_id"] = labels_df["session_id"].astype(int)
# Use final smoothed SOH if available; otherwise fall back to raw SOH
target_col = "soh_final" if "soh_final" in labels_df.columns else "soh"
label_dict = labels_df.set_index([vid_col, "session_id"])[target_col].to_dict()
clean_features_master = []
missing_labels = 0
# Join each feature block with its corresponding SOH label
for feat in raw_features:
vid = str(feat["vehicle_id"])
sid = int(feat["session_id"])
if (vid, sid) in label_dict:
feat_copy = feat.copy()
# Convert SOH fraction to capacity in Ah using nominal capacity of 155 Ah
feat_copy["measured_soh_ah"] = label_dict[(vid, sid)] * 155.0
clean_features_master.append(feat_copy)
else:
missing_labels += 1
print(f"Successfully synchronized {len(clean_features_master)} complete training samples.")
if missing_labels > 0:
print(f"Dropped {missing_labels} samples due to missing labels.")
# ============================================================================
63
# 3. DATASET CLASS (MULTIMODAL INPUT PIPELINE)
# ============================================================================
class OptimalBatteryDataset(Dataset):
"""
PyTorch dataset wrapping multimodal battery session data.
Each sample includes:
- voltage_map_2d: 2D voltage map image for the session.
- capacity_seq_1d: 1D sequence of capacity-related measurements.
- temperature_seq_1d: 1D sequence of temperatures aligned with capacity.
- point_features: engineered scalar features (e.g. HI statistics).
- measured_soh_ah: ground-truth capacity in Ah.
Preprocessing:
1. Voltage maps: NaNs replaced by 0; per-sample z-score normalization.
2. Sequential signals: divided by 100 for numerical scaling, NaNs→0.
3. Point features: train-set mean/std normalization.
4. Labels returned as SOH on [0, 1] scale (capacity / 155 Ah).
"""
def __init__(self, features, point_mean: torch.Tensor, point_std: torch.Tensor):
"""
Args:
features (list[dict]): List of feature dictionaries for each session.
point_mean (torch.Tensor): Train-set mean of point_features.
point_std (torch.Tensor): Train-set std of point_features.
"""
self.features = features
self.point_mean = point_mean
self.point_std = point_std
def __len__(self) -> int:
return len(self.features)
def __getitem__(self, idx: int):
"""
Build a single multimodal sample for the model.
Returns:
t_vmap (torch.FloatTensor): Voltage map, shape [1, H, W].
seq_1d (torch.FloatTensor): 1D sequence input, shape [2, T]
(capacity and temperature channels).
p_feat (torch.FloatTensor): Normalized static features, shape [F].
label (torch.FloatTensor): Normalized SOH target, shape [1].
"""
try:
feat = self.features[idx]
if feat is None:
raise ValueError(f"Feature at index {idx} is None")
# 1. Voltage map (Input A): NaN handling and z-score normalization
v_map = feat["voltage_map_2d"]
if np.isnan(v_map).any():
v_map = np.nan_to_num(v_map, nan=0.0)
# Per-sample standardization
v_map = (v_map - np.mean(v_map)) / (np.std(v_map) + 1e-6)
# Add a channel dimension (C=1) for Conv2D
t_vmap = torch.FloatTensor(v_map).unsqueeze(0)
64
# 2. Sequential data (Input B): capacity and temperature [2, T]
cap = np.nan_to_num(feat["capacity_seq_1d"] / 100.0, nan=0.0)
temp = np.nan_to_num(feat["temperature_seq_1d"] / 100.0, nan=0.0)
# Stacking on axis=0 yields two channels suitable for 1D sequence models
seq_1d = torch.FloatTensor(np.stack([cap, temp], axis=0))
# 3. Point features (Input C): NaN handling and train-set normalization
raw_p_feat = feat["point_features"]
if np.isnan(raw_p_feat).any():
raw_p_feat = np.nan_to_num(raw_p_feat, nan=0.0)
p_feat = (torch.FloatTensor(raw_p_feat) - self.point_mean) / self.point_std
# 4. Label: normalized SOH (0{1) derived from absolute capacity
label = torch.FloatTensor([feat["measured_soh_ah"] / 155.0])
return t_vmap, seq_1d, p_feat, label
except Exception:
# Safe fallback in case of corrupted data: return zero tensors with
# correct shapes so that training can continue.
return (
torch.zeros((1, 96, 96), dtype=torch.float32),
torch.zeros((2, 151), dtype=torch.float32),
torch.zeros((15,), dtype=torch.float32),
torch.zeros((1,), dtype=torch.float32),
)
def seed_worker(worker_id: int) -> None:
"""
Initialize worker-specific seeds for DataLoader workers.
This ensures that each DataLoader worker has its own reproducible random
state, derived from the global PyTorch seed.
"""
worker_seed = torch.initial_seed() % 2**32
np.random.seed(worker_seed)
random.seed(worker_seed)
# ==============================================================================
# 4. MODEL ARCHITECTURE: CONVNEXT ENCODER + MAMBA + MLP
# ==============================================================================
class ConvNeXtBlock(nn.Module):
"""
ConvNeXt-style residual block with depthwise convolution and MLP.
The block performs:
1. Depthwise convolution to mix spatial information per channel.
2. GroupNorm normalization.
3. Pointwise expansion, GELU activation, and pointwise projection.
4. Residual addition with the input feature map.
"""
def __init__(self, dim: int):
super().__init__()
self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
65
self.norm = nn.GroupNorm(1, dim)
self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
self.act = nn.GELU()
self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)
def forward(self, x: torch.Tensor) -> torch.Tensor:
return x + self.pwconv2(self.act(self.pwconv1(self.norm(self.dwconv(x)))))
class ConvNeXt_Encoder_Heavy(nn.Module):
"""
Hierarchical ConvNeXt-style encoder for 2D voltage map inputs.
The encoder:
- Progressively downsamples the spatial dimensions.
- Increases the channel dimensionality across stages.
- Applies ConvNeXt blocks at each resolution.
- Uses global average pooling and a linear layer to produce a fixed-size
embedding for downstream fusion.
"""
def __init__(self, input_channels: int = 1, output_dim: int = 128):
super().__init__()
self.stem = nn.Sequential(
nn.Conv2d(input_channels, 32, kernel_size=4, stride=4),
nn.GroupNorm(1, 32),
)
self.stage1 = self._make_layer(32, 64, blocks=3)
self.stage2 = self._make_layer(64, 128, blocks=3)
self.stage3 = self._make_layer(128, 256, blocks=3)
self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
# No dropout is used in this encoder; regularization is applied after fusion.
self.fc = nn.Linear(256, output_dim)
def _make_layer(self, in_dim: int, out_dim: int, blocks: int) -> nn.Sequential:
"""
Construct a downsampling stage followed by several ConvNeXt blocks.
"""
layers = [
nn.Sequential(
nn.GroupNorm(1, in_dim),
nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2),
)
]
layers.extend([ConvNeXtBlock(out_dim) for _ in range(blocks)])
return nn.Sequential(*layers)
def forward(self, x: torch.Tensor) -> torch.Tensor:
x = self.stage3(self.stage2(self.stage1(self.stem(x))))
x = self.avg_pool(x).view(x.size(0), -1)
return self.fc(x)
class HeavyMultiModalMambaSOH(nn.Module):
"""
Multimodal Mamba-based network for battery SOH regression.
Modalities:
66
- Image branch: ConvNeXt encoder for 2D voltage maps.
- Sequence branch: Mamba state space model with attention pooling applied
to capacity + temperature sequences.
- Static branch: MLP encoder for engineered point features.
The three modality-specific embeddings are layer-normalized and fused, and
the fused representation is passed through an MLP regression head to
produce a normalized SOH prediction.
"""
def __init__(self, seq_dim: int = 2, static_dim: int = 15, mamba_dim: int = 128):
super().__init__()
# Image (2D) encoder
self.cnn = ConvNeXt_Encoder_Heavy(input_channels=1, output_dim=mamba_dim)
# Sequence projection and Mamba SSM
# +1 channel accounts for appended temporal position encoding
self.input_proj = nn.Linear(seq_dim + 1, mamba_dim)
self.mamba = Mamba(d_model=mamba_dim, d_state=32, d_conv=4, expand=4)
# Attention over temporal dimension to obtain a fixed-length summary
self.attention = nn.Sequential(
nn.Linear(mamba_dim, mamba_dim // 2),
nn.Tanh(),
nn.Linear(mamba_dim // 2, 1),
nn.Softmax(dim=1),
)
# Static feature encoder
point_out_dim = 64
self.point_encoder = nn.Sequential(
nn.Linear(static_dim, 128),
nn.LayerNorm(128),
nn.GELU(),
# Dropout regularizes the static branch
nn.Dropout(0.05),
nn.Linear(128, point_out_dim),
)
self.norm_cnn = nn.LayerNorm(mamba_dim)
self.norm_mamba = nn.LayerNorm(mamba_dim)
self.norm_point = nn.LayerNorm(point_out_dim)
# Fusion dimension is the concatenation of the three modality embeddings
fusion_dim = mamba_dim + mamba_dim + point_out_dim
self.fusion_norm = nn.LayerNorm(fusion_dim)
# Regression head for SOH prediction
self.regressor = nn.Sequential(
nn.Linear(fusion_dim, 256),
nn.LayerNorm(256),
nn.GELU(),
nn.Dropout(0.02),
nn.Linear(256, 64),
nn.GELU(),
nn.Dropout(0.01),
nn.Linear(64, 1),
)
67
def forward(self, v_map: torch.Tensor, s_1d: torch.Tensor, p_feat: torch.Tensor):
"""
Forward pass.
Args:
v_map (torch.Tensor): Voltage map input of shape [B, 1, H, W].
s_1d (torch.Tensor): Sequence input of shape [B, C, T] (C=2).
p_feat (torch.Tensor): Static features, shape [B, F].
Returns:
torch.Tensor: Predicted normalized SOH of shape [B, 1].
"""
# --- Image branch ---
cnn_out = self.cnn(v_map)
# --- Sequence branch ---
# Rearrange to [B, T, C] for sequence processing
s_1d = s_1d.transpose(1, 2)
batch_size, seq_len, _ = s_1d.shape
# Append a temporal position channel in [0, 1] to encode order
time_pos = torch.linspace(0, 1, steps=seq_len, device=s_1d.device)
time_pos = time_pos.view(1, seq_len, 1).expand(batch_size, seq_len, 1)
s_1d = torch.cat([s_1d, time_pos], dim=2)
m_out = self.input_proj(s_1d)
m_out = self.mamba(m_out)
# Attention-based pooling along the temporal dimension
attn_weights = self.attention(m_out)
m_pooled = torch.sum(m_out * attn_weights, dim=1)
# --- Static feature branch ---
p_encoded = self.point_encoder(p_feat)
# --- Normalize and fuse representations ---
cnn_out = self.norm_cnn(cnn_out)
m_pooled = self.norm_mamba(m_pooled)
p_encoded = self.norm_point(p_encoded)
combined = torch.cat([cnn_out, m_pooled, p_encoded], dim=1)
combined = self.fusion_norm(combined)
if self.training:
# Apply light dropout at fusion level to regularize multimodal fusion
combined = nn.functional.dropout(combined, p=0.03)
return self.regressor(combined)
# Standard regression loss for normalized targets
criterion = nn.MSELoss()
# ==============================================================================
# 5. ENSEMBLE TRAINING MATRIX (5 SEEDS × 5 RUNS)
# ==============================================================================
# Master seeds define different train/test splits and initialization states
68
model_seeds = [44, 400, 300, 30, 20]
# For each master seed, run multiple independent initializations
run_suffixes = ["a", "b", "c", "d", "e"]
epochs = 100
accumulation_steps = 1 # Gradient accumulation (1 = standard per-batch update)
BATCH_SIZE = 128 # Global batch size for both training and evaluation
# Dictionary capturing per-seed and per-run performance
all_seed_results = {}
for current_seed in model_seeds:
print(f"\n" + "" * 70)
print(f"STARTING MASTER SEED: {current_seed}")
print("" * 70)
# Re-apply seed and re-split data for this master seed
set_seed(current_seed)
features_for_this_seed = list(clean_features_master)
random.shuffle(features_for_this_seed)
split_idx = int(0.8 * len(features_for_this_seed))
train_features = features_for_this_seed[:split_idx]
test_features = features_for_this_seed[split_idx:]
# Compute normalization statistics for static point features on the train split
train_points = np.stack(
[np.nan_to_num(f["point_features"], nan=0.0) for f in train_features]
)
point_mean = torch.FloatTensor(train_points.mean(axis=0))
point_std = torch.FloatTensor(train_points.std(axis=0))
# Avoid division by zero for degenerate features
point_std[point_std == 0] = 1.0
ensemble_mapes = []
for idx, suffix in enumerate(run_suffixes):
best_model_path = f"{DRIVE_PREFIX}/Mamba/best_MAMBA_{current_seed}_{suffix}.pth"
print(f"\n" + "=" * 60)
print(f"ENSEMBLE RUN ’{suffix.upper()}’ FOR MASTER SEED {current_seed}")
print("=" * 60)
# Run-specific seed for distinct weight initialization and batch ordering
run_seed = current_seed + (idx * 1000)
set_seed(run_seed)
# Construct datasets for this seed and its normalization statistics
train_dataset = OptimalBatteryDataset(train_features, point_mean, point_std)
test_dataset = OptimalBatteryDataset(test_features, point_mean, point_std)
# Generator ensures reproducible shuffling in the DataLoader
g = torch.Generator()
g.manual_seed(run_seed)
train_loader = DataLoader(
train_dataset,
69
batch_size=BATCH_SIZE,
shuffle=True,
drop_last=True,
worker_init_fn=seed_worker,
generator=g,
)
test_loader = DataLoader(
test_dataset,
batch_size=BATCH_SIZE,
shuffle=False,
)
# Initialize a fresh model for this run
model = HeavyMultiModalMambaSOH().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)
# OneCycleLR schedule: warm up to max_lr, then anneal
scheduler = OneCycleLR(
optimizer,
max_lr=0.001,
steps_per_epoch=len(train_loader),
epochs=epochs,
pct_start=0.35,
div_factor=25.0,
)
best_mape = float("inf")
patience = 55 # Early stopping patience (epochs)
epochs_no_improve = 0
# -----------------------------
# TRAINING AND VALIDATION LOOP
# -----------------------------
for epoch in range(1, epochs + 1):
# ---- Training phase ----
model.train()
train_mse_sum = 0.0
train_mape_sum = 0.0
valid_train_batches = 0
optimizer.zero_grad()
for i, (v_map, s_1d, p_feat, lbl) in enumerate(train_loader):
v_map, s_1d, p_feat, lbl = (
v_map.to(device),
s_1d.to(device),
p_feat.to(device),
lbl.to(device),
)
# Add small Gaussian noise to sequence inputs as data augmentation
if model.training:
noise = torch.randn_like(s_1d) * 0.01
s_1d = s_1d + noise
lbl = lbl.view(-1, 1)
out_norm = model(v_map, s_1d, p_feat).view(-1, 1)
70
# MSE loss in normalized SOH space
loss = criterion(out_norm, lbl) / accumulation_steps
# Skip numerically unstable batches
if torch.isnan(loss) or torch.isinf(loss):
continue
loss.backward()
# With accumulation_steps=1, gradients are updated every mini-batch
if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
optimizer.zero_grad()
# OneCycleLR is stepped once per batch
scheduler.step()
# Collect training RMSE and MAPE in capacity (Ah) space
with torch.no_grad():
out_real = out_norm * 155.0
lbl_real = lbl * 155.0
sq_err = torch.sum((out_real - lbl_real) ** 2)
train_mse_sum += sq_err.item()
abs_pct = (
torch.sum(
torch.abs(out_real - lbl_real) / (lbl_real + 1e-6)
)
* 100.0
)
train_mape_sum += abs_pct.item()
valid_train_batches += lbl.size(0)
avg_train_rmse = (
np.sqrt(train_mse_sum / valid_train_batches)
if valid_train_batches > 0
else 0.0
)
avg_train_mape = (
train_mape_sum / valid_train_batches if valid_train_batches > 0 else 0.0
)
# ---- Validation phase ----
model.eval()
val_mse_sum = 0.0
val_mape_sum = 0.0
valid_val_batches = 0
with torch.no_grad():
for v_map, s_1d, p_feat, lbl in test_loader:
v_map, s_1d, p_feat, lbl = (
v_map.to(device),
s_1d.to(device),
p_feat.to(device),
lbl.to(device),
71
)
lbl = lbl.view(-1, 1)
out_norm = model(v_map, s_1d, p_feat).view(-1, 1)
out_real = out_norm * 155.0
lbl_real = lbl * 155.0
sq_err = torch.sum((out_real - lbl_real) ** 2)
abs_pct = (
torch.sum(
torch.abs(out_real - lbl_real) / (lbl_real + 1e-6)
)
* 100.0
)
if not torch.isnan(sq_err):
val_mse_sum += sq_err.item()
val_mape_sum += abs_pct.item()
valid_val_batches += lbl.size(0)
avg_val_rmse = (
np.sqrt(val_mse_sum / valid_val_batches)
if valid_val_batches > 0
else 999.0
)
avg_val_mape = (
val_mape_sum / valid_val_batches if valid_val_batches > 0 else 999.0
)
# Early stopping based on validation MAPE
if avg_val_mape < best_mape:
best_mape = avg_val_mape
epochs_no_improve = 0
try:
torch.save(model.state_dict(), best_model_path)
saved_msg = "Saved Best"
except Exception:
saved_msg = "Save failed"
else:
epochs_no_improve += 1
saved_msg = f"No imp ({epochs_no_improve}/{patience})"
current_lr = scheduler.get_last_lr()[0]
print(
f"Epoch [{epoch}/{epochs}] | LR: {current_lr:.6f} | "
f"Train RMSE: {avg_train_rmse:.4f} Ah MAPE: {avg_train_mape:.2f}% | "
f"Val RMSE: {avg_val_rmse:.4f} Ah MAPE: {avg_val_mape:.2f}% | {saved_msg}"
)
if epochs_no_improve >= patience:
print(f"Early Stopping on Epoch {epoch}")
break
print(
f"Seed {current_seed}, Run {suffix.upper()} Complete. "
f"Best Val MAPE: {best_mape:.2f}%"
)
72
ensemble_mapes.append(best_mape)
# Free GPU and host memory before the next run in the ensemble
del model, optimizer, scheduler, train_loader, test_loader, train_dataset, test_dataset
gc.collect()
torch.cuda.empty_cache()
# Log ensemble results for this master seed
avg_seed_mape = np.mean(ensemble_mapes)
all_seed_results[current_seed] = {
"individual_runs": ensemble_mapes,
"ensemble_average": avg_seed_mape,
}
print(f"AVERAGE ENSEMBLE MAPE FOR SEED {current_seed}: {avg_seed_mape:.2f}%")
# ==============================================================================
# 6. SAVE FINAL ENSEMBLE RESULTS AS JSON
# ==============================================================================
results_dict = {
"timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
"model_type": "HeavyMultiModalMambaSOH",
"matrix_results": all_seed_results,
}
results_file_path = f"{DRIVE_PREFIX}/Mamba/25_run_ensemble_results.json"
with open(results_file_path, "w") as f:
json.dump(results_dict, f, indent=4)
print(f"\n All 25 runs completed! Results saved to: {results_file_path}")

In [ ]:
###
## Model 1 Evaluation
###

"""
Evaluation script for a multimodal Mamba-based battery SOH estimator.

This script:
1. Loads pre-extracted multimodal features and smoothed SOH labels.
2. Synchronizes feature blocks with session-level SOH targets.
3. Defines a PyTorch Dataset and a ConvNeXt-Mamba-MLP multimodal architecture.
4. Evaluates 25 trained models (5 seeds x 5 runs) on a fixed test split per seed.
5. Aggregates metrics into an academic-style table.
6. Generates publication-ready visualizations (scatter plots, residuals, trajectories,
   and attention weights) for the best-performing run.
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle
import sys
import random
import os
from torch.utils.data import Dataset, DataLoader
import math
from mamba_ssm import Mamba

# Plotting libraries for publication-quality figures
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# ============================================================================
# 0. REPRODUCIBILITY: GLOBAL SEEDING
# ============================================================================
def set_seed(seed: int) -> None:
    """
    Fix all relevant random seeds to ensure reproducible experiments.

    This function sets the random seed for Python, NumPy, and PyTorch (CPU and CUDA),
    and configures cuDNN for deterministic behaviour where possible. Using a fixed
    seed is essential for replicable train/test splits and model initialization
    across multiple runs.
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# 1. DATA PATHS, DEVICE SELECTION, AND LOADING
# ============================================================================
# Base folders for Google Drive in Colab
DRIVE_PREFIX = "/content/drive/MyDrive/Final_Year_Project"
DRIVE_PREFIX2 = "/content/drive/MyDrive"

# Pre-extracted multimodal feature blocks and smoothed SOH labels
INPUT_DATA_PATH = f"{DRIVE_PREFIX}/multimodal_features_synced2.pkl"
LABELS_CSV_PATH = f"{DRIVE_PREFIX2}/session_level_soh_smoothed_final2.csv"

# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading extracted features from {INPUT_DATA_PATH}...")
try:
    # Each element of raw_features is a dict with multimodal feature fields
    with open(INPUT_DATA_PATH, "rb") as f:
        raw_features = pickle.load(f)
    print(f" Loaded {len(raw_features)} feature blocks.")
except Exception as e:
    sys.exit(f"\n CRITICAL ERROR loading features: {e}")

print(f"Loading SOH labels from {LABELS_CSV_PATH}...")
try:
    # Session-level smoothed SOH targets
    with open(LABELS_CSV_PATH, "rb") as f:
        labels_df = pd.read_csv(f)
    print(f" Loaded {len(labels_df)} target SOH labels.")
except Exception as e:
    sys.exit(f"\n CRITICAL ERROR loading labels: {e}")

# ============================================================================
# 2. SYNCHRONIZE FEATURES WITH SESSION-LEVEL LABELS
# ============================================================================
print(" Merging features with target labels...")

# Harmonize vehicle identifier column name and types
vid_col = "vehicle_id_split" if "vehicle_id_split" in labels_df.columns else "vehicle_id"
labels_df[vid_col] = labels_df[vid_col].astype(str)
labels_df["session_id"] = labels_df["session_id"].astype(int)

# Harmonize SOH column name; target is on [0, 1] scale
target_col = "soh_final" if "soh_final" in labels_df.columns else "soh"
label_dict = labels_df.set_index([vid_col, "session_id"])[target_col].to_dict()

clean_features = []
for feat in raw_features:
    # Match each feature block to its corresponding SOH label
    vid = str(feat["vehicle_id"])
    sid = int(feat["session_id"])

    if (vid, sid) in label_dict:
        feat_copy = feat.copy()
        # Convert SOH fraction to absolute capacity (Ah) using nominal capacity 155 Ah
        feat_copy["measured_soh_ah"] = label_dict[(vid, sid)] * 155.0
        clean_features.append(feat_copy)

# ============================================================================
# 3. DATASET DEFINITION AND MODEL ARCHITECTURE
# ============================================================================
class OptimalBatteryDataset(Dataset):
    """
    PyTorch dataset for multimodal battery SOH estimation.

    Each sample contains:
    - voltage_map_2d: 2D voltage map (HI image) for a single charging session.
    - capacity_seq_1d: 1D sequence of capacity-related HI values over time.
    - temperature_seq_1d: 1D temperature sequence aligned with capacity.
    - point_features: 15 engineered scalar health indicators (HI points).
    - measured_soh_ah: target capacity in Ah, derived from SOH x nominal capacity.

    The dataset:
    1. Normalizes each voltage map per sample (zero mean, unit variance).
    2. Scales 1D sequences (capacity and temperature) by 1/100 for numerical stability.
    3. Applies train-set mean/std normalization to point_features.
    4. Returns SOH on [0, 1] scale as the learning target.
    """

    def __init__(self, features, point_mean: torch.Tensor, point_std: torch.Tensor):
        """
        Args:
            features (list[dict]): List of feature dictionaries for each session.
            point_mean (torch.Tensor): Mean of point_features computed on the train set.
            point_std (torch.Tensor): Std of point_features computed on the train set.
        """
        self.features = features
        self.point_mean = point_mean
        self.point_std = point_std

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int):
        """
        Construct one multimodal sample suitable for the model.

        Returns:
            t_vmap (torch.FloatTensor): Voltage map, shape [1, H, W].
            seq_1d (torch.FloatTensor): 1D sequence input, shape [2, T]
                                        (capacity and temperature channels).
            p_feat (torch.FloatTensor): Normalized point features, shape [F].
            label (torch.FloatTensor): Normalized SOH target, shape [1].
        """
        try:
            feat = self.features[idx]

            # --- 2D voltage map preprocessing ---
            v_map = feat["voltage_map_2d"]

            # Replace NaNs with 0 to avoid numerical issues
            if np.isnan(v_map).any():
                v_map = np.nan_to_num(v_map, nan=0.0)

            # Per-sample standardization (zero mean, unit variance)
            v_map = (v_map - np.mean(v_map)) / (np.std(v_map) + 1e-6)

            # Add channel dimension expected by ConvNeXt (C=1)
            t_vmap = torch.FloatTensor(v_map).unsqueeze(0)

            # --- 1D temporal sequences (capacity and temperature) ---
            cap = np.nan_to_num(feat["capacity_seq_1d"] / 100.0, nan=0.0)
            temp = np.nan_to_num(feat["temperature_seq_1d"] / 100.0, nan=0.0)

            # Stack as [channels, time]; two channels: capacity and temperature
            seq_1d = torch.FloatTensor(np.stack([cap, temp], axis=0))

            # --- Engineered point features ---
            raw_p_feat = feat["point_features"]
            if np.isnan(raw_p_feat).any():
                raw_p_feat = np.nan_to_num(raw_p_feat, nan=0.0)

            # Train-set normalization for static features
            p_feat = (torch.FloatTensor(raw_p_feat) - self.point_mean) / self.point_std

            # --- Target SOH (normalized by nominal capacity) ---
            label = torch.FloatTensor([feat["measured_soh_ah"] / 155.0])

            return t_vmap, seq_1d, p_feat, label

        except Exception:
            # In case of unexpected data issues, return a safe all-zero placeholder
            # to keep batch shapes consistent and avoid crashing the evaluation loop.
            return (
                torch.zeros((1, 96, 96), dtype=torch.float32),
                torch.zeros((2, 151), dtype=torch.float32),
                torch.zeros((15,), dtype=torch.float32),
                torch.zeros((1,), dtype=torch.float32),
            )


class ConvNeXtBlock(nn.Module):
    """
    ConvNeXt-style residual block with depthwise convolution and pointwise MLP.

    The block applies:
    1. Depthwise convolution (spatial mixing within each channel).
    2. GroupNorm for channel-wise normalization.
    3. Pointwise expansion (4x) and GELU non-linearity.
    4. Pointwise projection back to the original channel dimension.
    5. Residual connection around the entire sub-block.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.GroupNorm(1, dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pwconv2(self.act(self.pwconv1(self.norm(self.dwconv(x)))))


class ConvNeXt_Encoder_Heavy(nn.Module):
    """
    Hierarchical ConvNeXt-style encoder for the 2D voltage map modality.

    The encoder progressively downsamples the spatial resolution while increasing
    the channel dimensionality, and finally applies global average pooling and
    a linear projection to obtain a compact representation.
    """

    def __init__(self, input_channels: int = 1, output_dim: int = 128):
        super().__init__()

        # Initial stem: coarse downsampling of the input voltage map
        self.stem = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=4, stride=4),
            nn.GroupNorm(1, 32),
        )

        # Three ConvNeXt stages with increasing feature dimensions
        self.stage1 = self._make_layer(32, 64, blocks=3)
        self.stage2 = self._make_layer(64, 128, blocks=3)
        self.stage3 = self._make_layer(128, 256, blocks=3)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # No dropout is used in this CNN branch; regularization is applied later.
        self.fc = nn.Linear(256, output_dim)

    def _make_layer(self, in_dim: int, out_dim: int, blocks: int) -> nn.Sequential:
        """
        Construct a downsampling + ConvNeXt-block stage.

        The first operation halves the spatial resolution and increases the
        channel dimension, followed by several ConvNeXt residual blocks.
        """
        layers = [
            nn.Sequential(
                nn.GroupNorm(1, in_dim),
                nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2),
            )
        ]
        layers.extend([ConvNeXtBlock(out_dim) for _ in range(blocks)])
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stage3(self.stage2(self.stage1(self.stem(x))))
        # Global average pooling followed by a linear projection
        return self.fc(self.avg_pool(x).view(x.size(0), -1))


class HeavyMultiModalMambaSOH(nn.Module):
    """
    Multimodal Mamba-based architecture for battery SOH regression.

    Modalities:
    - 2D voltage maps (processed by a ConvNeXt encoder).
    - 1D temporal sequences (capacity + temperature processed by a Mamba SSM).
    - Static engineered point features (processed by an MLP encoder).

    The model:
    1. Extracts a CNN embedding from the voltage map.
    2. Augments the 1D sequence with a temporal position encoding and processes
       it via a Mamba state space layer and attention pooling.
    3. Encodes static point features with a shallow MLP.
    4. Fuses all three modalities and predicts normalized SOH via an MLP head.
    """

    def __init__(self, seq_dim: int = 2, static_dim: int = 15, mamba_dim: int = 128):
        super().__init__()

        # Image (2D voltage map) encoder
        self.cnn = ConvNeXt_Encoder_Heavy(input_channels=1, output_dim=mamba_dim)

        # Sequence projection and Mamba state space model
        # +1 channel for appended temporal position feature
        self.input_proj = nn.Linear(seq_dim + 1, mamba_dim)
        self.mamba = Mamba(d_model=mamba_dim, d_state=32, d_conv=4, expand=4)

        # Attention module for pooling along the temporal dimension
        self.attention = nn.Sequential(
            nn.Linear(mamba_dim, mamba_dim // 2),
            nn.Tanh(),
            nn.Linear(mamba_dim // 2, 1),
            nn.Softmax(dim=1),
        )

        # Static point-feature encoder
        point_out_dim = 64
        self.point_encoder = nn.Sequential(
            nn.Linear(static_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            # Light dropout to mitigate overfitting on static features
            nn.Dropout(0.05),
            nn.Linear(128, point_out_dim),
        )

        # LayerNorm for each branch before fusion
        self.norm_cnn = nn.LayerNorm(mamba_dim)
        self.norm_mamba = nn.LayerNorm(mamba_dim)
        self.norm_point = nn.LayerNorm(point_out_dim)

        # Fusion of three modalities (CNN + Mamba + static)
        fusion_dim = mamba_dim + mamba_dim + point_out_dim
        self.fusion_norm = nn.LayerNorm(fusion_dim)

        # Regression head from fused representation to scalar SOH prediction
        self.regressor = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.02),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.01),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        v_map: torch.Tensor,
        s_1d: torch.Tensor,
        p_feat: torch.Tensor,
        return_attention: bool = False,
    ):
        """
        Forward pass for multimodal SOH prediction.

        Args:
            v_map (torch.Tensor): Voltage map input of shape [B, 1, H, W].
            s_1d (torch.Tensor): Sequence input of shape [B, C, T]
                                 where C=2 (capacity, temperature).
            p_feat (torch.Tensor): Static point-feature input of shape [B, F].
            return_attention (bool): If True, also return temporal attention weights.

        Returns:
            out (torch.Tensor): Predicted normalized SOH of shape [B, 1].
            (Optional) attn_weights (torch.Tensor): Attention weights of shape [B, T, 1].
        """
        # --- Image branch ---
        cnn_out = self.cnn(v_map)

        # --- Sequence branch ---
        # Rearrange to [B, T, C] for sequence processing
        s_1d = s_1d.transpose(1, 2)
        batch_size, seq_len, _ = s_1d.shape

        # Append a normalized temporal position channel in [0, 1] to encode order
        time_pos = torch.linspace(0, 1, steps=seq_len, device=s_1d.device)
        time_pos = time_pos.view(1, seq_len, 1).expand(batch_size, seq_len, 1)
        s_1d = torch.cat([s_1d, time_pos], dim=2)

        # Project to Mamba dimension and apply state space model
        m_out = self.input_proj(s_1d)
        m_out = self.mamba(m_out)

        # Attention-based pooling along time to obtain a fixed-size sequence summary
        attn_weights = self.attention(m_out)
        m_pooled = torch.sum(m_out * attn_weights, dim=1)

        # --- Static feature branch ---
        p_encoded = self.point_encoder(p_feat)

        # --- Normalization + multimodal fusion ---
        cnn_out = self.norm_cnn(cnn_out)
        m_pooled = self.norm_mamba(m_pooled)
        p_encoded = self.norm_point(p_encoded)

        combined = torch.cat([cnn_out, m_pooled, p_encoded], dim=1)
        combined = self.fusion_norm(combined)

        if self.training:
            # Apply a small dropout at fusion level only during training
            combined = nn.functional.dropout(combined, p=0.03)

        out = self.regressor(combined)

        if return_attention:
            return out, attn_weights
        return out

# ==============================================================================
# 4. EVALUATE 25 MODELS (5 SEEDS x 5 RUNS)
# ==============================================================================
# Random seeds correspond to different train/test splits and initialization states
model_seeds = [44, 400, 300, 30, 20]

# Suffixes distinguish multiple runs for each seed
run_suffixes = ["a", "b", "c", "d", "e"]

# Seed-level aggregated metrics
seed_mapes = []
seed_rmses = []
seed_max_errs = []

# Per-run metrics across all 25 evaluations
all_run_mapes = []
all_run_rmses = []
all_run_maxerr = []

# Track the single best run for downstream plotting and interpretability analysis
global_best_mape = float("inf")
best_run_preds = None
best_run_lbls = None
best_test_features = None
best_model_state = None
best_dataset_params = None

run_seed_tracker = []  # For boxplots across seeds

print("\n Evaluating 25 models (5 seeds x 5 runs)...\n")

for seed in model_seeds:
    print(f"\n{'='*60}")
    print(f" Evaluating SEED {seed}")
    print(f"{'='*60}")

    # Reconstruct the exact train/test split used for this seed
    set_seed(seed)
    features_for_this_seed = list(clean_features)
    random.shuffle(features_for_this_seed)

    split_idx = int(0.8 * len(features_for_this_seed))
    train_features = features_for_this_seed[:split_idx]
    test_features = features_for_this_seed[split_idx:]

    # Compute normalization statistics for point_features on the train split only
    train_points = np.stack(
        [np.nan_to_num(f["point_features"], nan=0.0) for f in train_features]
    )
    point_mean = torch.FloatTensor(train_points.mean(axis=0))
    point_std = torch.FloatTensor(train_points.std(axis=0))

    # Avoid division by zero for constant features
    point_std[point_std == 0] = 1.0

    # Construct test dataset and loader using train-set statistics
    test_dataset = OptimalBatteryDataset(test_features, point_mean, point_std)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    # Metrics for this specific seed (averaged over its 5 runs)
    this_seed_mapes = []
    this_seed_rmses = []
    this_seed_max_errs = []

    for idx, suffix in enumerate(run_suffixes):
        model_path = f"{DRIVE_PREFIX}/Mamba/best_MAMBA_{seed}_{suffix}.pth"
        print(f" Loading Run '{suffix.upper()}': {model_path}")

        # Instantiate and load model weights for this run
        model = HeavyMultiModalMambaSOH().to(device)
        try:
            model.load_state_dict(
                torch.load(model_path, map_location=device, weights_only=True)
            )
        except Exception as e:
            print(f" Could not load {model_path}: {e}. Skipping.")
            continue

        model.eval()

        run_preds = []
        run_lbls = []
        val_mape_sum = 0.0
        valid_val_batches = 0

        with torch.no_grad():
            for v_map, s_1d, p_feat, lbl in test_loader:
                v_map, s_1d, p_feat, lbl = (
                    v_map.to(device),
                    s_1d.to(device),
                    p_feat.to(device),
                    lbl.to(device),
                )
                lbl = lbl.view(-1, 1)
                out_norm = model(v_map, s_1d, p_feat).view(-1, 1)

                # Convert normalized outputs back to capacity in Ah
                out_real = out_norm * 155.0
                lbl_real = lbl * 155.0

                # MAPE computed consistently with the training script:
                # sum over absolute percentage errors, then divide by N
                abs_pct = (
                    torch.sum(torch.abs(out_real - lbl_real) / (lbl_real + 1e-6)) * 100.0
                )
                if not torch.isnan(abs_pct):
                    val_mape_sum += abs_pct.item()
                    valid_val_batches += lbl.size(0)

                run_preds.append(out_real.cpu().numpy())
                run_lbls.append(lbl_real.cpu().numpy())

        # --- Per-run metrics ---
        run_mape = val_mape_sum / valid_val_batches if valid_val_batches > 0 else 999.0

        run_preds_arr = np.vstack(run_preds)
        run_lbls_arr = np.vstack(run_lbls)
        abs_errors = np.abs(run_preds_arr - run_lbls_arr)

        # RMSE expressed as percentage of nominal capacity
        run_rmse_pct = (np.sqrt(np.mean(abs_errors**2)) / 155.0) * 100.0

        # Maximum absolute percentage error over all samples in this run
        run_max_err = np.max((abs_errors / (run_lbls_arr + 1e-6)) * 100.0)

        # Track the single best run globally (lowest MAPE) for interpretability plots
        if run_mape < global_best_mape:
            global_best_mape = run_mape
            best_run_preds = run_preds_arr.flatten()
            best_run_lbls = run_lbls_arr.flatten()
            best_test_features = test_features
            best_model_state = model.state_dict()
            best_dataset_params = (point_mean, point_std)

        this_seed_mapes.append(run_mape)
        this_seed_rmses.append(run_rmse_pct)
        this_seed_max_errs.append(run_max_err)

        all_run_mapes.append(run_mape)
        all_run_rmses.append(run_rmse_pct)
        all_run_maxerr.append(run_max_err)

        run_seed_tracker.append({"Seed": str(seed), "MAPE": run_mape, "RMSE": run_rmse_pct})

        print(
            f"   Run {suffix.upper()} → MAPE: {run_mape:.2f}% | "
            f"RMSE: {run_rmse_pct:.2f}% | Max Error: {run_max_err:.2f}%"
        )

    # Per-seed mean metrics aggregated over all successful runs for this seed
    seed_avg_mape = np.mean(this_seed_mapes)
    seed_avg_rmse = np.mean(this_seed_rmses)
    seed_avg_max = np.mean(this_seed_max_errs)

    seed_mapes.append(seed_avg_mape)
    seed_rmses.append(seed_avg_rmse)
    seed_max_errs.append(seed_avg_max)

    print(
        f"\n Seed {seed} Average → MAPE: {seed_avg_mape:.2f}% | "
        f"RMSE: {seed_avg_rmse:.2f}% | Max Error: {seed_avg_max:.2f}%"
    )

# ==============================================================================
# 5. AGGREGATED METRICS AND ACADEMIC TABLE
# ==============================================================================
# Overall averages and standard deviations across all 25 runs
avg_mape = np.mean(all_run_mapes)
std_mape = np.std(all_run_mapes)
avg_rmse = np.mean(all_run_rmses)
std_rmse = np.std(all_run_rmses)
avg_max = np.mean(all_run_maxerr)
std_max = np.std(all_run_maxerr)
best_mape = np.min(all_run_mapes)

print(f"\n\n{'-'*60}")
print(" FINAL RESULTS ACROSS ALL 25 RUNS")
print(f"{'-'*60}")
print(f" Best Single Run MAPE : {best_mape:.2f}%")
print(f" Average MAPE         : {avg_mape:.2f}% ± {std_mape:.3f}%")
print(f" Average RMSE         : {avg_rmse:.2f}% ± {std_rmse:.3f}%")
print(f" Average Max Error    : {avg_max:.2f}% ± {std_max:.3f}%")

# Build a per-seed summary table with mean and per-seed standard deviation
table_data = []
for i, seed in enumerate(model_seeds):
    table_data.append(
        {
            ("Seed", ""): str(seed),
            ("MAPE (%)", "Average"): f"{seed_mapes[i]:.2f}",
            ("MAPE (%)", "Std."): f"{np.std([all_run_mapes[i * 5 + j] for j in range(5)]):.3f}",
            ("RMSE (%)", "Average"): f"{seed_rmses[i]:.2f}",
            ("RMSE (%)", "Std."): f"{np.std([all_run_rmses[i * 5 + j] for j in range(5)]):.3f}",
            ("Max Error (%)", "Avg"): f"{seed_max_errs[i]:.2f}",
        }
    )

# Append an overall row summarizing all runs
table_data.append(
    {
        ("Seed", ""): "Overall",
        ("MAPE (%)", "Average"): f"{avg_mape:.2f}",
        ("MAPE (%)", "Std."): f"{std_mape:.3f}",
        ("RMSE (%)", "Average"): f"{avg_rmse:.2f}",
        ("RMSE (%)", "Std."): f"{std_rmse:.3f}",
        ("Max Error (%)", "Avg"): f"{avg_max:.2f}",
    }
)

df_results = pd.DataFrame(table_data)
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns)

# In a notebook, display a styled table; otherwise, fall back to plain text
try:
    styled_df = df_results.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left")]}]
    )
    display(styled_df)
except NameError:
    print(df_results)

# ==============================================================================
# 6. GENERATE PUBLICATION-READY PLOTS
# ==============================================================================
print("\n Generating academic plots based on the best run...")

# Use the globally best run (lowest MAPE) for scatter plots and trajectories
preds_cap = best_run_preds
actuals_cap = best_run_lbls

# Convert capacity back to SOH (0-1 scale)
preds_soh = preds_cap / 155.0
actuals_soh = actuals_cap / 155.0

# ---------------------------------------------------------
# Plot 1a: Predicted vs Actual SOH (Dynamically Scaled)
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(x=actuals_soh, y=preds_soh, alpha=0.5, edgecolor=None, color="#1f77b4")

# Dynamic limits with a small buffer to avoid clipping extreme points
min_soh = min(actuals_soh.min(), preds_soh.min()) - 0.02
max_soh = max(actuals_soh.max(), preds_soh.max()) + 0.02
plt.xlim(min_soh, max_soh)
plt.ylim(min_soh, max_soh)

plt.plot([min_soh, max_soh], [min_soh, max_soh], "r--", lw=2, label="Ideal Fit (y=x)")
plt.xlabel("Measured SOH", fontweight="bold")
plt.ylabel("Predicted SOH", fontweight="bold")
plt.title("Best Run: Predicted vs. Actual SOH", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Plot 1b: Predicted vs Actual Capacity (Dynamically Scaled)
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(x=actuals_cap, y=preds_cap, alpha=0.5, edgecolor=None, color="#2ca02c")

# Dynamic limits with a 2 Ah buffer around the point cloud
min_cap = min(actuals_cap.min(), preds_cap.min()) - 2.0
max_cap = max(actuals_cap.max(), preds_cap.max()) + 2.0
plt.xlim(min_cap, max_cap)
plt.ylim(min_cap, max_cap)

plt.plot([min_cap, max_cap], [min_cap, max_cap], "r--", lw=2, label="Ideal Fit (y=x)")
plt.xlabel("Measured Capacity (Ah)", fontweight="bold")
plt.ylabel("Predicted Capacity (Ah)", fontweight="bold")
plt.title("Best Run: Predicted vs. Actual Capacity", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Plot 2: Residual Error Distribution (Capacity)
# ---------------------------------------------------------
residuals_cap = preds_cap - actuals_cap

plt.figure(figsize=(8, 6))
sns.histplot(residuals_cap, kde=True, bins=40, color="#ff7f0e")
plt.axvline(x=0, color="red", linestyle="--", lw=2)
plt.xlabel("Residual Error (Predicted − Actual Capacity in Ah)", fontweight="bold")
plt.ylabel("Frequency", fontweight="bold")
plt.title("Distribution of Residual Errors (Capacity)", fontweight="bold")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Plot 3: Boxplots of MAPE Across Seeds (Stability Analysis)
# ---------------------------------------------------------
df_metrics = pd.DataFrame(run_seed_tracker)

plt.figure(figsize=(10, 6))
sns.boxplot(x="Seed", y="MAPE", data=df_metrics, palette="Set2")
sns.stripplot(x="Seed", y="MAPE", data=df_metrics, color=".25", size=6, jitter=True)
plt.xlabel("Random Seed", fontweight="bold")
plt.ylabel("Test MAPE (%)", fontweight="bold")
plt.title("Model Stability: MAPE Distribution Across 25 Runs", fontweight="bold")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Plot 4: Capacity Degradation Trajectories for Selected Vehicles
# ---------------------------------------------------------
df_traj = pd.DataFrame(
    {
        "vehicle_id": [f["vehicle_id"] for f in best_test_features],
        "session_id": [f["session_id"] for f in best_test_features],
        "actual_cap": actuals_cap,
        "pred_cap": preds_cap,
    }
)

# Select a few vehicles with the longest test-set histories for visualization
top_vehicles = df_traj["vehicle_id"].value_counts().head(2).index.tolist()

if top_vehicles:
    plt.figure(figsize=(12, 6))
    colors = ["#1f77b4", "#d62728"]

    global_min_traj = float("inf")
    global_max_traj = float("-inf")

    for i, vid in enumerate(top_vehicles):
        v_data = df_traj[df_traj["vehicle_id"] == vid].sort_values("session_id")

        plt.plot(
            v_data["session_id"],
            v_data["actual_cap"],
            label=f"Vehicle {vid} (Actual)",
            color=colors[i],
            linestyle="-",
            marker="o",
            markersize=4,
        )
        plt.plot(
            v_data["session_id"],
            v_data["pred_cap"],
            label=f"Vehicle {vid} (Predicted)",
            color=colors[i],
            linestyle="--",
            alpha=0.7,
        )

        # Track global limits to scale the y-axis consistently
        local_min = min(v_data["actual_cap"].min(), v_data["pred_cap"].min())
        local_max = max(v_data["actual_cap"].max(), v_data["pred_cap"].max())
        global_min_traj = min(global_min_traj, local_min)
        global_max_traj = max(global_max_traj, local_max)

    # Dynamic scaling for the trajectory plot with a small buffer
    plt.ylim(global_min_traj - 2.0, global_max_traj + 2.0)

    plt.xlabel("Session ID (Cycles/Time)", fontweight="bold")
    plt.ylabel("Capacity (Ah)", fontweight="bold")
    plt.title("Capacity Degradation Trajectory Tracking", fontweight="bold")
    plt.legend()
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Plot 5: Temporal Attention Weights (Interpretability)
# ---------------------------------------------------------
if best_model_state is not None:
    # Reconstruct the best-performing model with its trained parameters
    best_model = HeavyMultiModalMambaSOH().to(device)
    best_model.load_state_dict(best_model_state)
    best_model.eval()

    # Build a dataset object for the best test split using stored normalization
    p_mean, p_std = best_dataset_params
    best_dataset = OptimalBatteryDataset(best_test_features, p_mean, p_std)

    # Inspect attention for the first sample in the best test set
    v_map, s_1d, p_feat, lbl = best_dataset[0]
    v_map = v_map.unsqueeze(0).to(device)
    s_1d = s_1d.unsqueeze(0).to(device)
    p_feat = p_feat.unsqueeze(0).to(device)

    with torch.no_grad():
        _, attn_weights = best_model(v_map, s_1d, p_feat, return_attention=True)

    # attn_weights has shape [1, Seq_Len, 1]; squeeze to 1D for plotting
    attn_weights = attn_weights.squeeze().cpu().numpy()

    plt.figure(figsize=(12, 4))
    plt.plot(attn_weights, color="#2ca02c", lw=2)
    plt.fill_between(range(len(attn_weights)), attn_weights, color="#2ca02c", alpha=0.3)

    # Dynamic y-axis scaling for attention weights
    plt.ylim(0, max(attn_weights) * 1.1)

    plt.xlabel("Sequence Step (Time)", fontweight="bold")
    plt.ylabel("Attention Weight", fontweight="bold")
    plt.title("Mamba Temporal Attention Mechanism (Sample Interpretability)", fontweight="bold")
    plt.tight_layout()
    plt.show()

print("All dynamically scaled plots generated successfully!")
